In [36]:
import numpy as np
import pandas as pd

In [37]:
credit_score = np.random.randint(300, 850, 1000)
DTI_Ratio = np.random.uniform(0, 1, 1000)
Default = ((credit_score < 600) & (DTI_Ratio > 0.5)).astype(int)
flip = np.random.random(1000) < 0.1
Default = np.where(flip, 1 - Default, Default)
loans = pd.DataFrame({'credit_score': credit_score,
                      'DTI_Ratio': DTI_Ratio,
                      'Default': Default})

In [38]:
def split(df, depth):
    lowest_gini = 1
    gini_column = None
    decision_rule_value = None

    for column in df.drop(columns='Default').columns:
        for value in df[column].unique():
            try:
                left = df[df[column] < value]
                p_left_left = len(left[left['Default'] == 1]) / len(left)
                p_left_right = len(left[left['Default'] == 0]) / len(left)
                Gini_left = 1 - p_left_right**2 - p_left_left**2

                right = df[df[column] >= value]
                p_right_left = len(right[right['Default'] == 1]) / len(right)
                p_right_right = len(right[right['Default'] == 0]) / len(right)
                Gini_right = 1 - p_right_left**2 - p_right_right**2

                Gini = (len(left) / len(df)) * Gini_left + (len(right) / len(df)) * Gini_right

                if Gini < lowest_gini:
                    lowest_gini = Gini
                    gini_column = column
                    decision_rule_value = value

            except ZeroDivisionError:
                continue

    return lowest_gini, gini_column, decision_rule_value, depth

In [ ]:
def tree_dict(subsets, depth):
    decision_rule_dict = {}
    next_subsets = []
    rule_num = 0

    for subset in subsets:
        if subset is None or len(subset) == 0:
            next_subsets.append(None)
            next_subsets.append(None)
            continue

        _, gini_column, decision_rule_value, _ = split(subset, depth)

        if gini_column is None:
            majority_class = 1 if subset['Default'].mean() >= 0.5 else 0
            decision_rule_dict[f'Decision Rule {rule_num}'] = (
                f'Leaf Node: Majority Class = {majority_class}')
            next_subsets.append(None)
            next_subsets.append(None)
        else:
            left  = subset[subset[gini_column] < decision_rule_value]
            right = subset[subset[gini_column] >= decision_rule_value]
            decision_rule_dict[f'Decision Rule {rule_num}'] = (
                f'{gini_column} < {decision_rule_value}')
            next_subsets.append(left)  
            next_subsets.append(right) 

        rule_num += 1

    return decision_rule_dict, next_subsets

In [40]:
def build_tree(df, y, maximum_depth, classification_regression='Classification'):
    tree = {}
    depth = 0
    subsets = [df]

    while depth <= maximum_depth:
        if depth == maximum_depth:
            # Force all remaining subsets into leaf nodes
            decision_rule_dict = {}
            for rule_num, subset in enumerate(s for s in subsets if s is not None and len(s) > 0):
                if classification_regression == 'Classification':
                    majority_class = 1 if subset[y].mean() >= 0.5 else 0
                    decision_rule_dict[f'Decision Rule {rule_num}'] = (f'Leaf Node: Majority Class = {majority_class}')
                if classification_regression == 'Regression':
                    prediction = subset[y].mean()
                    decision_rule_dict[f'Decision Rule {rule_num}'] = (f'Leaf Node: Average = {prediction}')

            tree[f'Depth {depth}'] = decision_rule_dict
            break

        decision_rule_dict, subsets = tree_dict(subsets, depth)
        tree[f'Depth {depth}'] = decision_rule_dict
        depth += 1

    return tree

In [41]:
def predict(df, tree):
    maximum_depth = len(tree.keys())

    df['y_pred'] = None
    df = df.reset_index()
    index = df['index']
    for row in range(len(df)):
        decision_rule_options = 0
        found_prediction = False
        
        for depth_num in range(maximum_depth):
            try:
                rule_num = decision_rule_options
                column, split_value = tree[f'Depth {depth_num}'][f'Decision Rule {rule_num}'].split(' < ')
                split_value = float(split_value)
                
                leaf_offset = sum(
                    1 for i in range(rule_num)
                    if 'Leaf Node' in tree[f'Depth {depth_num}'][f'Decision Rule {i}']
                )
                
                if df[column].iloc[row] < split_value:
                    decision_rule_options = rule_num * 2 - leaf_offset * 2
                else:
                    decision_rule_options = rule_num * 2 + 1 - leaf_offset * 2

            except ValueError:
                prediction = tree[f'Depth {depth_num}'][f'Decision Rule {rule_num}'].split(' = ')[1]
                df.loc[row, 'y_pred'] = prediction
                found_prediction = True

            except KeyError:
                continue

            if found_prediction:
                break

    df = df.set_index(index).drop(columns='index')
    
    return df

In [45]:
def calculate_metrics(df, y_true, y_pred):
    df[y_true]       = df[y_true].astype(int)
    df[y_pred]        = df[y_pred].astype(int)

    true_positives      = ((df[y_true] == 1) & (df[y_pred] == 1)).sum()
    false_positives     = ((df[y_true] == 0) & (df[y_pred] == 1)).sum()

    true_negatives      = ((df[y_true] == 0) & (df[y_pred] == 0)).sum()
    false_negatives     = ((df[y_true] == 1) & (df[y_pred] == 0)).sum()

    positive_support    = (df[y_true] == 1).sum()
    negative_support    = (df[y_true] == 0).sum()
    total               = len(df)

    precision_1         = true_positives / (true_positives + false_positives)
    recall_1            = true_positives / (true_positives + false_negatives)
    f1_1                = 2 * (precision_1 * recall_1) / (precision_1 + recall_1)

    precision_0         = true_negatives / (true_negatives + false_negatives)
    recall_0            = true_negatives / (true_negatives + false_positives)
    f1_0                = 2 * (precision_0 * recall_0) / (precision_0 + recall_0)

    accuracy = (true_positives + true_negatives) / total
    
    macro_precision     = (precision_1 + precision_0) / 2
    macro_recall        = (recall_1 + recall_0) / 2
    macro_f1            = (f1_1 + f1_0) / 2

    weighted_precision  = (precision_1 * positive_support + precision_0 * negative_support) / total
    weighted_recall     = (recall_1 * positive_support + recall_0 * negative_support) / total
    weighted_f1         = (f1_1 * positive_support + f1_0 * negative_support) / total

    metric_dict =   {'0': {'precision': precision_0,
                         'recall':    recall_0,
                         'f1':        f1_0,
                         'support':   negative_support},
                    '1': {'precision': precision_1,
                          'recall':    recall_1,
                          'f1':        f1_1,
                          'support':   positive_support},
                    'Accuracy': {'precision': '',
                                 'recall':    '',
                                 'f1':        accuracy,
                                 'support':   total},
                    'macro avg': {'precision': macro_precision,
                                  'recall':    macro_recall,
                                  'f1':        macro_f1,
                                  'support':   total},
                    'weighted avg': {'precision': weighted_precision,
                                     'recall':    weighted_recall,
                                     'f1':        weighted_f1,
                                     'support':   total}}
    
    metric_df = pd.DataFrame(metric_dict).T
    
    return metric_dict, metric_df

In [46]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(loans, test_size=0.3, random_state=42)

tree = build_tree(train, 'Default', 5)
pred_df = predict(test, tree)

calculate_metrics(pred_df, 'Default', 'y_pred')[1]

,precision,recall,f1,support
0,0.915094,0.92381,0.919431,210.0
1,0.818182,0.8,0.808989,90.0
Accuracy,,,0.886667,300
macro avg,0.866638,0.861905,0.86421,300.0
weighted avg,0.886021,0.886667,0.886299,300.0


In [47]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.metrics import classification_report

clf = DecisionTreeClassifier(max_depth=5)
clf = clf.fit(X = train[['DTI_Ratio', 'credit_score']], y = train['Default'])
prediction = clf.predict(test[['DTI_Ratio', 'credit_score']])

test['y_pred'] = prediction
print(classification_report(test['Default'], test['y_pred']))

              precision    recall  f1-score   support

           0       0.91      0.92      0.92       210
           1       0.82      0.79      0.80        90

    accuracy                           0.88       300
   macro avg       0.86      0.86      0.86       300
weighted avg       0.88      0.88      0.88       300

